# All Agents Streaming Workflow

This notebook demonstrates streamed calls to every published GAS agent.

The main workflow creates a county-level 2021 population dataset for Pennsylvania, inspects it, projects it, analyzes it, creates raster/vector/statistical outputs, and generates static maps and web mapping apps.

The PASDA agent is included as an independent streamed data-discovery step.

## Install GAS Client SDK

This notebook uses the published `gas-client` package from PyPI. Run this cell once in a new notebook environment.


In [ ]:
%pip install -q gas-client


In [ ]:
from pathlib import Path
from urllib.parse import urljoin

from IPython.display import Image, HTML, display

project_root = Path.cwd()
if project_root.name == "examples_for_using_gas_services":
    project_root = project_root.parent


from gas_client import GasClient


## User Settings

In [ ]:
server_url = "http://127.0.0.1:4042"

openai_api_key = "YOUR_OPENAI_API_KEY"
us_census_demography_key = "YOUR_US_CENSUS_DEMOGRAPHY_KEY"

poll_timeout = 2400

## Helpers

In [ ]:

def first_artifact_url(task_result, preferred_extensions=None):
    artifacts = task_result.get("outputs", {}).get("artifacts", [])
    preferred_extensions = preferred_extensions or []

    for extension in preferred_extensions:
        for artifact in artifacts:
            url = artifact.get("url")
            filename = artifact.get("filename") or artifact.get("name") or url or ""
            if url and str(filename).lower().endswith(extension.lower()):
                return url

    for artifact in artifacts:
        if artifact.get("url"):
            return artifact["url"]

    raise RuntimeError("The task returned no artifact URL.")


def absolute_url(url):
    if url.startswith("/"):
        return urljoin(server_url, url)
    return url


def display_visual_artifacts(task_result):
    artifacts = task_result.get("outputs", {}).get("artifacts", [])
    for artifact in artifacts:
        url = artifact.get("url")
        filename = artifact.get("filename") or artifact.get("name") or ""
        if not url:
            continue

        display_url = absolute_url(url)
        lower_name = str(filename).lower()
        if lower_name.endswith(".png"):
            display(Image(url=display_url))
        elif lower_name.endswith(".html"):
            display(HTML(f'<a href="{display_url}" target="_blank">Open interactive HTML artifact</a>'))


## Create Client and Agent Handles

In [ ]:
client = GasClient(server_url, openai_api_key=openai_api_key)

pasda_agent = client.agent("pasda_agent")
retrieval_agent = client.agent("geospatial_data_retrieval_agent")
inspection_agent = client.agent("geospatial_data_inspection_agent")
projection_agent = client.agent("map_projection_agent")
vector_agent = client.agent("vector_analysis_agent")
raster_agent = client.agent("raster_agent")
statistics_agent = client.agent("spatial_statistics_agent")
mapping_agent = client.agent("mapping_agent")
web_mapping_app_agent = client.agent("web_mapping_app_agent")

## 1. PASDA Agent: Streamed Data Discovery

In [ ]:

pasda_task = None
for event in pasda_agent.execute_task(
    "Find and download Pennsylvania county boundary data from PASDA.",
    mode="stream",
    artifact_delivery="URL",
    timeout=poll_timeout,
):
    client.print_stream_event(event)
    if event.get("event") == "task_result":
        pasda_task = event.get("payload")

if pasda_task is None:
    raise RuntimeError("The stream ended before returning a task_result event.")

client.print_task_summary(pasda_task)


## 2. Geospatial Data Retrieval Agent: Streamed Census County Population Dataset

In [ ]:

retrieval_task = None
for event in retrieval_agent.execute_task(
    "Download PA county population data for 2021.",
    mode="stream",
    artifact_delivery="URL",
    credentials={
        "source_credentials": {
            "US_Census_demography": {
                "key": us_census_demography_key,
            }
        }
    },
    timeout=poll_timeout,
):
    client.print_stream_event(event)
    if event.get("event") == "task_result":
        retrieval_task = event.get("payload")

if retrieval_task is None:
    raise RuntimeError("The stream ended before returning a task_result event.")

client.print_task_summary(retrieval_task)
county_population_url = first_artifact_url(retrieval_task, preferred_extensions=[".geojson", ".gpkg"])
county_population_url


## 3. Geospatial Data Inspection Agent: Streamed Dataset Inspection

In [ ]:

inspection_task = None
for event in inspection_agent.execute_task(
    "Inspect this county population dataset. Report CRS, geometry type, feature count, useful columns, missing values, and whether it is ready for mapping and spatial analysis.",
    mode="stream",
    input_datasets=[county_population_url],
    artifact_delivery="URL",
    timeout=poll_timeout,
):
    client.print_stream_event(event)
    if event.get("event") == "task_result":
        inspection_task = event.get("payload")

if inspection_task is None:
    raise RuntimeError("The stream ended before returning a task_result event.")

client.print_task_summary(inspection_task)


## 4. Map Projection Agent: Streamed Lambert Conformal Conic Projection

In [ ]:

projection_task = None
for event in projection_agent.execute_task(
    "Project the data to Lambert Conformal Conic",
    mode="stream",
    input_datasets=[county_population_url],
    artifact_delivery="URL",
    timeout=poll_timeout,
):
    client.print_stream_event(event)
    if event.get("event") == "task_result":
        projection_task = event.get("payload")

if projection_task is None:
    raise RuntimeError("The stream ended before returning a task_result event.")

client.print_task_summary(projection_task)
projected_counties_url = first_artifact_url(projection_task, preferred_extensions=[".geojson", ".gpkg"])
projected_counties_url


## 5. Vector Analysis Agent: Streamed Vector Transformation

In [ ]:

vector_task = None
for event in vector_agent.execute_task(
    (
        "Using the projected county population dataset, calculate each county's area in square kilometers, "
        "calculate population density, keep the original geometry, and return one GeoJSON dataset."
    ),
    mode="stream",
    input_datasets=[projected_counties_url],
    artifact_delivery="URL",
    timeout=poll_timeout,
):
    client.print_stream_event(event)
    if event.get("event") == "task_result":
        vector_task = event.get("payload")

if vector_task is None:
    raise RuntimeError("The stream ended before returning a task_result event.")

client.print_task_summary(vector_task)
density_counties_url = first_artifact_url(vector_task, preferred_extensions=[".geojson", ".gpkg"])
density_counties_url


## 6. Raster Agent: Streamed Vector-to-Raster Workflow

In [ ]:

raster_task = None
for event in raster_agent.execute_task(
    (
        "Rasterize the Pennsylvania county polygons to a GeoTIFF. "
        "Use the 'population_density' field as pixel values. "
        "Use 100-meter pixels. "
        "The output GeoTIFF must include CRS, affine transform, nodata value, and one float32 band."
    ),
    mode="stream",
    input_datasets=[density_counties_url],
    artifact_delivery="URL",
    timeout=poll_timeout,
):
    client.print_stream_event(event)
    if event.get("event") == "task_result":
        raster_task = event.get("payload")

if raster_task is None:
    raise RuntimeError("The stream ended before returning a task_result event.")

client.print_task_summary(raster_task)
raster_url = first_artifact_url(raster_task, preferred_extensions=[".tif", ".tiff"])
raster_url


## 7. Spatial Statistics Agent: Streamed Autocorrelation Analysis

In [ ]:

statistics_task = None
for event in statistics_agent.execute_task(
    (
        "Check the spatial autocorrelation of county population density. Use queen contiguity weights if appropriate, "
        "and return a concise report plus any useful diagnostic artifact."
    ),
    mode="stream",
    input_datasets=[density_counties_url],
    artifact_delivery="URL",
    timeout=poll_timeout,
):
    client.print_stream_event(event)
    if event.get("event") == "task_result":
        statistics_task = event.get("payload")

if statistics_task is None:
    raise RuntimeError("The stream ended before returning a task_result event.")

client.print_task_summary(statistics_task)


## 8. Mapping Agent: Streamed Static Choropleth Map

In [ ]:

mapping_task = None
for event in mapping_agent.execute_task(
    (
        "Create a county-level choropleth map for the contiguous United States showing 2021 population density. "
        "Use a quantile classification scheme with 5 classes. Use a sequential palette where the lowest class is visibly light blue, not white. "
        "Draw missing values in light gray and use thin county outlines. The input dataset is already in Lambert Conformal Conic."
    ),
    mode="stream",
    input_datasets=[density_counties_url],
    artifact_delivery="URL",
    timeout=poll_timeout,
):
    client.print_stream_event(event)
    if event.get("event") == "task_result":
        mapping_task = event.get("payload")

if mapping_task is None:
    raise RuntimeError("The stream ended before returning a task_result event.")

client.print_task_summary(mapping_task)
display_visual_artifacts(mapping_task)


## 9. Web Mapping App Agent: Streamed Web Mapping App

In [ ]:

interactive_task = None
for event in web_mapping_app_agent.execute_task(
    (
        "Create an interactive county-level map for the contiguous United States using the population density field. "
        "Include hover popups with county name, 2021 population, and population density. Use a visible sequential color scheme."
    ),
    mode="stream",
    input_datasets=[density_counties_url],
    artifact_delivery="URL",
    timeout=poll_timeout,
):
    client.print_stream_event(event)
    if event.get("event") == "task_result":
        interactive_task = event.get("payload")

if interactive_task is None:
    raise RuntimeError("The stream ended before returning a task_result event.")

client.print_task_summary(interactive_task)
display_visual_artifacts(interactive_task)


## Final Artifact URLs

In [ ]:
all_tasks = {
    "pasda": pasda_task,
    "retrieval": retrieval_task,
    "inspection": inspection_task,
    "projection": projection_task,
    "vector_analysis": vector_task,
    "raster": raster_task,
    "spatial_statistics": statistics_task,
    "mapping": mapping_task,
    "interactive_mapping": interactive_task,
}

for name, task_result in all_tasks.items():
    print("\n" + name)
    for artifact in task_result.get("outputs", {}).get("artifacts", []):
        if artifact.get("url"):
            print("-", artifact.get("filename") or artifact.get("name"), artifact["url"])